In [ ]:
from training_env.market import Market
from models.algorithms.ppo import train
from typing import List, Tuple
from .settings import DIR
import random

2025-10-21 12:37:30.978174: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [2]:
seed = 123
random.seed(seed)

In [3]:
def train_test_split(files: List[str], test_size: float) -> Tuple[List[str], List[str]]:
    size = len(files)

    test_files= random.sample(files, int(test_size * size))
    train_files = [file for file in files if file not in test_files]


    return train_files, test_files

In [ ]:
import os


files = []
path_to_data = os.path.join(DIR, 'data', 'ready')

for file in os.listdir(path_to_data):
    path = os.path.join(path_to_data, file)
    if os.path.isfile(path):
        files.append(path)


train_files, test_files = train_test_split(files, test_size=0.2)

In [5]:
print(len(test_files))
print(len(train_files))
print(f'{len(files)}={len(test_files)}+{len(train_files)}')

3040
12164
15204=3040+12164


In [6]:
train_env = Market(training_files=train_files, initial_cash=10_000, slippage=0.03, broker_fee=0.01, lam=0.1, seed=seed)
test_env = Market(training_files=test_files, initial_cash=10_000, slippage=0.03, broker_fee=0.01, lam=0.1, seed=seed)

In [ ]:
import optuna


def objective(trial: optuna.trial.Trial):
    hidden = trial.suggest_int(name='hidden', low=8, high=128, step=8)

    actor_lr = trial.suggest_float(name='actor_lr', low=2e-5, high=5e-4)
    critic_lr = trial.suggest_float(name='critic_lr', low=2e-5, high=5e-4)

    gamma = trial.suggest_float(name='gamma', low=0.8, high=1.0)
    lam = trial.suggest_float(name='gamma', low=0.8, high=1.0)

    opt_epochs = trial.suggest_int(name='opt_epochs', low=4, high=11)

    clip_ratio = trial.suggest_float(name='clip_ratio', low=0.1, high=0.4, step=0.1)

    c1 = trial.suggest_float(name='c1', low=0.5, high=1.0)
    c2 = trial.suggest_float(name='c2', low=0.0001, high=0.01)

    batch_size = trial.suggest_int(name='batch_size', low=256, high=2048, step=128)

    params = {
        'env': train_env,
        'eval_env': test_env,
        'hidden_shape': hidden,
        'actor_lr': actor_lr,
        'critic_lr': critic_lr,
        'advantage_type': 'gae',
        'gamma': gamma,
        'lam': lam,
        'clip_ratio': clip_ratio,
        'opt_epochs': opt_epochs,
        'c1': c1,
        'c2': c2,
        'batch_size': batch_size,
        'display_stat': False,
        'eval_episodes': 6,
        'total_steps': 12500,
        'save_path': '',
        'seed': seed,
    }

    objective, rewards = train(**params)

    return objective, rewards


study = optuna.create_study(directions=['maximize', 'maximize'])
study.optimize(objective, n_trials=50)


[I 2025-10-21 12:37:42,823] A new study created in memory with name: no-name-45428083-e312-43c6-bf19-0bab354882fe
I0000 00:00:1761039463.153006   16618 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 6117 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 4060 Laptop GPU, pci bus id: 0000:01:00.0, compute capability: 8.9
Steps:  24%|███████▍                       | 2999/12500 [01:34<04:57, 31.89it/s]
[W 2025-10-21 12:39:17,311] Trial 0 failed with parameters: {'hidden': 72, 'actor_lr': 0.00047016215668213817, 'critic_lr': 0.0001580119044755669, 'gamma': 0.9086251959992048, 'opt_epochs': 10, 'clip_ratio': 0.4, 'c1': 0.887806876167371, 'c2': 0.004873919394056561, 'batch_size': 1920} because of the following error: KeyboardInterrupt().
Traceback (most recent call last):
  File "/home/danil/Documents/ML/Project/Untitled_Trading_Bot/.venv/lib/python3.12/site-packages/optuna/study/_optimize.py", line 201, in _run_trial
    value_or_values = func(trial)
  

KeyboardInterrupt: 

In [ ]:
best_params = study.best_params
best_obj, best_rewards = study.best_value

print('Best parameters', best_params, sep='\n')
print(f'Best objective: {best_obj:.3f} | Best mean reward: {best_rewards:.3f}')